# Customer Segmentation for Targeted Marketing
### RFM Analysis + Unsupervised Clustering (K-Means, with DBSCAN comparison)

**Business objective:** Segment an e-commerce customer base into actionable behavioral
groups using RFM (Recency, Frequency, Monetary) analysis and clustering, enabling
targeted retention campaigns and estimated revenue recovery from at-risk customers.

**Author:** *(your name)*
**Dataset:** Synthetic e-commerce transaction data (mirrors the schema of the UCI
*Online Retail II* dataset). See `data/generate_data.py` for generation logic, or
swap in the real Online Retail II dataset — see README for instructions.

---
## Contents
1. Load & Clean Data
2. RFM Feature Engineering
3. Transform & Scale
4. Choose Optimal k (Elbow + Silhouette)
5. Fit K-Means & Compare with DBSCAN
6. Profile & Name Clusters (Business Layer)
7. Business Impact Estimate
8. Visualizations
9. Conclusions & Recommendations


In [1]:
import sys, os
sys.path.append(os.getcwd())
from analysis_core import *
import pandas as pd
pd.set_option('display.max_columns', None)


## 1. Load & Clean Data

Remove cancelled orders, guest checkouts (missing Customer ID), and invalid rows.

In [2]:
df = load_and_clean()
df.head()

Raw rows: 102,060  ->  Cleaned rows: 98,514  (3,546 removed)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,540159,22720,SET OF 3 CAKE TINS PANTRY DESIGN,1,2024-01-01 08:05:00,5.37,12498,Ireland,5.37
1,544511,47566,PARTY BUNTING,4,2024-01-01 08:21:00,5.17,13029,France,20.68
2,568494,22383,LUNCH BAG SUKI DESIGN,6,2024-01-01 08:29:00,1.61,15837,Australia,9.66
3,550591,21212,PACK OF 72 RETRO SPOT CAKE CASES,16,2024-01-01 09:06:00,0.58,13721,United Kingdom,9.28
4,547492,21931,JUMBO STORAGE BAG SUKI,9,2024-01-01 09:10:00,2.06,13374,United Kingdom,18.54


## 2. RFM Feature Engineering

For every customer, compute:
- **Recency** — days since their last purchase
- **Frequency** — number of distinct orders
- **Monetary** — total amount spent

In [3]:
rfm = build_rfm(df)
rfm.describe()

,Recency,Frequency,Monetary
count,3994.000000,3994.000000,3994.000000
mean,68.158988,8.461442,621.654767
std,96.476515,6.732679,949.917718
min,1.000000,1.000000,5.420000
25%,3.000000,3.000000,85.342500
50%,12.000000,8.000000,301.635000
75%,132.000000,12.000000,624.542500
max,366.000000,30.000000,4786.250000


## 3. Transform & Scale

RFM distributions are heavily right-skewed (a small number of customers spend far more than the rest). We log-transform then standardize before clustering, since K-Means is distance-based and sensitive to scale/skew.

In [4]:
rfm_scaled = transform_scale(rfm)
rfm_scaled[:5]

array([[-0.09902945, -1.60060928, -1.29785771],
       [-1.30035973,  1.57337476,  1.85265366],
       [-0.17617291, -0.02631124, -0.68779642],
       [-1.06611534,  0.54168072, -0.33815429],
       [ 1.20158176,  1.01254327,  0.6991098 ]])

## 4. Choose Optimal k

We use the **Elbow Method** (inertia) alongside the **Silhouette Score** to select the number of clusters. Silhouette is more decisive here since the elbow can be ambiguous.

In [5]:
best_k, inertia, sil_scores = choose_k(rfm_scaled)
print('Silhouette scores by k:', {k: round(v,3) for k,v in sil_scores.items()})
print('Selected k =', best_k)

Silhouette scores by k: {2: 0.414, 3: 0.438, 4: 0.503, 5: 0.543, 6: 0.53, 7: 0.519, 8: 0.496}
Selected k = 5


![Elbow and Silhouette](../outputs/plots/01_k_selection.png)

## 5. Fit K-Means & Compare with DBSCAN

We fit K-Means with the selected `k`, and run DBSCAN as a comparison to check whether a density-based approach finds materially different structure (useful to justify the choice of algorithm in an interview).

In [6]:
rfm, comparison = fit_models(rfm, rfm_scaled, best_k)
comparison

{'kmeans_k': 5,
 'kmeans_silhouette': 0.543,
 'dbscan_clusters_found': 2,
 'dbscan_noise_points': 0,
 'dbscan_silhouette': 0.411}

**Why K-Means over DBSCAN here:** DBSCAN found far fewer, less business-actionable
clusters on this data because RFM segments tend to form density-similar,
roughly-convex groups rather than well-separated, high-density regions with a clear
"noise" class. K-Means' higher silhouette score, plus clusters that map cleanly onto
recognizable business segments, makes it the better choice for this use case.

## 6. Profile & Name Clusters — the Business Layer

This is the step that turns raw cluster IDs into something a marketing team can act on. Clusters are ranked by an aggregate RFM score and labeled using Recency/Frequency signals so every segment gets a distinct, meaningful name.

In [7]:
summary = name_clusters(rfm)
summary[['Recency','Frequency','Monetary','Count','Segment','Recommended_Action']]

,Recency,Frequency,Monetary,Count,Segment,Recommended_Action
KMeans_Cluster,,,,,,
0,7.2,9.4,424.7,1268,Potential Loyalists,"Loyalty program invite, cross-sell recommendat..."
1,274.7,1.0,34.4,480,Lapsed / Dormant,Reactivation campaign or sunset from active ma...
2,1.5,22.3,3025.1,482,High-Value Loyalists,"VIP loyalty perks, early access to new product..."
3,13.6,2.7,82.4,949,New / Low-Engagement Customers,"Onboarding series, first-purchase follow-up, w..."
4,144.2,9.9,480.4,815,At-Risk / Churning,"Win-back email campaign, personalized discount..."


## 7. Business Impact Estimate

We quantify the revenue opportunity in the **At-Risk / Churning** segment — the single most actionable number for a stakeholder conversation.

In [8]:
impact = business_impact(rfm, summary)
impact

{'total_customers': 3994,
 'total_historical_revenue': np.float64(2482889.14),
 'at_risk_customer_count': 815,
 'at_risk_pct_of_base': 20.4,
 'at_risk_historical_value': np.float64(391526.0),
 'estimated_recoverable_revenue_low': np.float64(58728.9),
 'estimated_recoverable_revenue_high': np.float64(78305.2),
 'assumption': "15-20% win-back conversion rate on at-risk segment's historical spend"}

> **Business takeaway:** roughly a fifth of the customer base has drifted into an
> at-risk state while still representing meaningful historical spend. Even a modest
> win-back conversion rate translates into a non-trivial recoverable revenue range —
> this is the number to lead with when pitching the segmentation to a marketing team.

## 8. Visualizations

In [9]:
make_visuals(rfm, summary)
print('Plots saved to ../outputs/plots/')

/home/claude/customer-segmentation-project/notebook/analysis_core.py:239: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=order["Segment"], y=order["Count"], palette="viridis")


Plots saved to ../outputs/plots/


**Segment sizes:**

![Segment sizes](../outputs/plots/02_segment_sizes.png)

**Recency vs Monetary by segment:**

![Recency vs Monetary](../outputs/plots/03_recency_vs_monetary.png)

**Segment profile radar chart:**

![Radar](../outputs/plots/04_radar_segment_profiles.png)

## 9. Conclusions & Recommendations

- Customers were successfully segmented into **5 behaviorally distinct groups** using RFM + K-Means (silhouette score ≈ 0.54, indicating well-separated clusters).
- Each segment maps to a concrete marketing action (see table in Section 6) — this turns an ML exercise into an operational marketing plan, not just a clustering demo.
- The **At-Risk / Churning** segment is the highest-priority target: it's large enough to matter and represents real historical value that is currently disengaging.
- **Next steps for a production version:**
  - Automate this pipeline to refresh segments monthly/quarterly as customer behavior shifts.
  - A/B test the recommended actions (e.g., win-back email vs. discount) per segment and measure actual recovery rate against the estimate here.
  - Feed segment labels into the CRM/marketing automation tool so campaigns can be triggered automatically.
  - Extend features beyond RFM (e.g., product category preferences, channel, return rate) for finer-grained personalization.

Outputs (`rfm_clusters.csv`, `cluster_summary.csv`, `business_impact.json`) are saved to `/outputs` for downstream use, e.g. by the Streamlit app in `/app`.
